# 04 — Feature Engineering
## Goal
Create features that are available **when the customer places the order**.

### Allowed
- rider attributes already known
- restaurant/customer coordinates already known
- order date
- order time
- weather/traffic/context known at order time

### Not allowed
`Time_Order_picked` — pickup has already happened, so using it would leak future information into the prediction.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../outputs/cleaned/zomato_cleaned_raw_level.csv")
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

def parse_time_to_minutes(series):
    text = series.astype("string").str.strip()
    result = pd.Series(np.nan, index=series.index, dtype="float64")

    colon_mask = text.str.contains(":", na=False)

    parsed = pd.to_datetime(
        text.where(colon_mask),
        format="%H:%M",
        errors="coerce"
    )
    valid = parsed.notna()
    result.loc[valid] = parsed.loc[valid].dt.hour * 60 + parsed.loc[valid].dt.minute

    numeric_fraction = pd.to_numeric(text.where(~colon_mask), errors="coerce")
    valid_fraction = numeric_fraction.between(0, 1, inclusive="both")
    result.loc[valid_fraction] = numeric_fraction.loc[valid_fraction] * 1440

    return result

def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * radius * np.arcsin(np.sqrt(a))

# Distance between restaurant and customer.
df["distance_km"] = haversine_km(
    df["Restaurant_latitude"],
    df["Restaurant_longitude"],
    df["Delivery_location_latitude"],
    df["Delivery_location_longitude"]
)

# Order-time features: available at prediction time.
df["order_minutes"] = parse_time_to_minutes(df["Time_Orderd"])
df["order_hour"] = np.floor(df["order_minutes"] / 60)
df["order_day_of_week"] = df["Order_Date"].dt.dayofweek
df["is_weekend"] = df["order_day_of_week"].isin([5, 6]).astype(int)
df["is_peak_hour"] = (
    df["order_hour"].between(12, 14) |
    df["order_hour"].between(19, 22)
).astype(int)

df["time_of_day"] = pd.cut(
    df["order_hour"],
    bins=[-1, 6, 11, 16, 19, 24],
    labels=["Night", "Morning", "Afternoon", "Evening", "Night_2"]
).astype("string").replace({"Night_2": "Night"})

# Drop raw columns that are identifiers, target, or leakage-prone.
drop_cols = [
    "ID",
    "Delivery_person_ID",
    "Restaurant_latitude",
    "Restaurant_longitude",
    "Delivery_location_latitude",
    "Delivery_location_longitude",
    "Order_Date",
    "Time_Orderd",
    "Time_Order_picked",
    "order_minutes"
]

model_df = df.drop(columns=drop_cols)

# Keep the target separate.
target = "Time_taken (min)"
X = model_df.drop(columns=[target])
y = model_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

X shape: (45584, 16)
y shape: (45584,)


,Delivery_person_Age,Delivery_person_Ratings,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,distance_km,order_hour,order_day_of_week,is_weekend,is_peak_hour,time_of_day
0,36.0,4.2,Fog,Jam,2.0,Snack,motorcycle,3.0,No,Metropolitian,10.280582,21.0,5,1,1,Night
1,21.0,4.7,Stormy,High,1.0,Meal,motorcycle,1.0,No,Metropolitian,6.242319,14.0,6,1,1,Afternoon
2,23.0,4.7,Sandstorms,Medium,1.0,Drinks,scooter,1.0,No,Metropolitian,13.787860,17.0,4,0,0,Evening
3,34.0,4.3,Sandstorms,Low,0.0,Buffet,motorcycle,0.0,No,Metropolitian,2.930258,9.0,6,1,0,Morning
4,24.0,4.7,Fog,Jam,1.0,Snack,scooter,1.0,No,Metropolitian,19.396618,19.0,0,0,1,Evening


In [2]:
# Inspect engineered features
display(
    model_df[
        ["distance_km", "order_hour", "order_day_of_week",
         "is_weekend", "is_peak_hour", "time_of_day", target]
    ].describe(include="all")
)

,distance_km,order_hour,order_day_of_week,is_weekend,is_peak_hour,time_of_day,Time_taken (min)
count,45584.000000,43853.000000,45584.000000,45584.000000,45584.000000,43853,45584.000000
unique,NaN,NaN,NaN,NaN,NaN,4,NaN
top,NaN,NaN,NaN,NaN,NaN,Night,NaN
freq,NaN,NaN,NaN,NaN,NaN,18291,NaN
mean,9.735119,17.632089,2.997521,0.274987,0.466830,NaN,26.293963
std,5.608034,4.533547,1.969709,0.446513,0.498904,NaN,9.384298
min,1.465067,8.000000,0.000000,0.000000,0.000000,NaN,10.000000
25%,4.663345,15.000000,1.000000,0.000000,0.000000,NaN,19.000000
50%,9.220148,19.000000,3.000000,0.000000,0.000000,NaN,26.000000
75%,13.681492,21.000000,5.000000,1.000000,1.000000,NaN,32.000000


In [3]:
Path("../outputs/cleaned").mkdir(parents=True, exist_ok=True)
model_df.to_csv("../outputs/cleaned/zomato_model_dataset.csv", index=False)

print("Saved model-ready dataset:", model_df.shape)

Saved model-ready dataset: (45584, 17)


## Why distance?

The raw GPS coordinates describe two locations, but the model can learn more directly from a distance feature.

We use Haversine distance because latitude/longitude are coordinates on a sphere. The result is an approximate straight-line distance in kilometres — not road-network distance.

## Why no pickup time?

Because our prediction moment is **order placement**. Pickup time is future information at that moment. Including it would make offline metrics look better than what would be possible in real use.